In [2]:
#!/usr/bin/env python

"""
ETS baseline (statsforecast.AutoETS) for remaining-time prediction.

Assumptions:
- You already ran:
    1) warm-up trimming + train/val/test split
    2) prefix builder

So in `PREFIX_INPUT_FOLDER` you have files like:
    ..._warm10_train_prefix.csv
    ..._warm10_val_prefix.csv
    ..._warm10_test_prefix.csv

For each *base* log, we:
  1) Build a case-level cycle-time series from the TRAIN prefixes.
  2) Fit AutoETS (Hyndman ETS) on that series.
  3) Forecast horizon = (#val cases + #test cases).
  4) Map forecasted total cycle times to val/test cases.
  5) For every prefix row, compute:
        D_hat_ets  = predicted total cycle time for that case
        R_hat_ets  = max(D_hat_ets - elapsed_time, 0)

Outputs:
- One CSV per val/test input, with extra ETS columns, written to OUTPUT_FOLDER.
"""

import os
import glob

import numpy as np
import pandas as pd

from statsforecast import StatsForecast
from statsforecast.models import AutoETS


# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
PREFIX_INPUT_FOLDER = r"out/251110/prefix_datasets"        # where *_prefix.csv live
OUTPUT_FOLDER       = r"out/251110/prefix_with_ets"        # where we'll save predictions

# AutoETS config: no seasonality, automatic error/trend choice
AUTOETS_MODEL      = "ZZN"   # automatic error/trend, no seasonal component
AUTOETS_SEASON_LEN = 1       # no meaningful seasonality in case sequence
# ------------------------------------------------------------------


def load_prefix_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = [
        "case_id",
        "timestamp",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
    ]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"{path} missing required column: {col}")

    return df


def build_case_table(df_prefix: pd.DataFrame) -> pd.DataFrame:
    """
    From a prefix dataset, get one row per case with:
    - case_id
    - case_start_time
    - case_complete_time
    - cycle_time = complete - start

    We take the first row per case (they all share the same start/complete times).
    """
    cols = ["case_id", "case_start_time", "case_complete_time"]
    cases = (
        df_prefix
        .sort_values(["case_id", "timestamp"])
        .drop_duplicates(subset=["case_id"], keep="first")
        [cols]
        .copy()
    )
    cases["cycle_time"] = cases["case_complete_time"] - cases["case_start_time"]
    # sort by start time to define the time-series order
    cases = cases.sort_values("case_start_time").reset_index(drop=True)
    return cases


def build_ts_df(cases: pd.DataFrame, unique_id: str) -> pd.DataFrame:
    """
    Build the statsforecast-style time series dataframe for AutoETS:

        columns: [unique_id, ds, y]

    where:
      - unique_id: same string for all rows (each scenario is one series)
      - ds: integer time index (0,1,2,...)
      - y: cycle_time
    """
    n = len(cases)
    ts_df = pd.DataFrame({
        "unique_id": unique_id,
        "ds": np.arange(n),
        "y": cases["cycle_time"].to_numpy(),
    })
    return ts_df


def fit_autoets(train_ts: pd.DataFrame) -> StatsForecast:
    """
    Fit AutoETS on the training time series.
    """
    sf = StatsForecast(
        models=[AutoETS(model=AUTOETS_MODEL, season_length=AUTOETS_SEASON_LEN)],
        freq=1,   # arbitrary integer index frequency
    )
    sf.fit(df=train_ts)
    return sf


def forecast_cycle_times(sf: StatsForecast, h: int) -> np.ndarray:
    """
    Forecast h future cycle times using AutoETS.
    Returns a 1D numpy array of length h.
    """
    y_hat = sf.predict(h=h)
    # column name is the model class name, here "AutoETS"
    preds = y_hat["AutoETS"].to_numpy()
    if len(preds) != h:
        raise RuntimeError(f"Expected {h} forecasts, got {len(preds)}")
    return preds


def add_ets_predictions_to_prefix(
    df_prefix: pd.DataFrame,
    case_table: pd.DataFrame,
    D_hat: np.ndarray,
    case_ids_in_order: np.ndarray,
) -> pd.DataFrame:
    """
    Attach case-level ETS predictions to every prefix row.

    Inputs:
    - df_prefix: prefix-level dataframe (val or test).
    - case_table: dataframe with one row per case.
    - D_hat: array of predicted total cycle times, aligned with case_ids_in_order.
    - case_ids_in_order: np.array of case_ids (sorted) that correspond to D_hat.

    Output:
    - df_prefix copy with:
        * D_hat_ets
        * R_hat_ets = max(D_hat_ets - elapsed_time, 0)
    """
    df_prefix = df_prefix.copy()

    # map from case_id -> predicted total cycle time
    case_pred_df = pd.DataFrame({
        "case_id": case_ids_in_order,
        "D_hat_ets": D_hat,
    })

    # merge predictions into prefix rows
    df_prefix = df_prefix.merge(case_pred_df, on="case_id", how="left")

    # compute predicted remaining time
    df_prefix["R_hat_ets"] = np.clip(
        df_prefix["D_hat_ets"] - df_prefix["elapsed_time"],
        a_min=0.0,
        a_max=None,
    )

    return df_prefix


def process_scenario(train_path: str):
    """
    For one "base" scenario, find its train/val/test prefix files,
    fit AutoETS on train, predict for val+test, and write out new CSVs.
    """
    base_name = os.path.basename(train_path)
    base_root = base_name.replace("_warm10_train_prefix.csv", "")
    print(f"\n=== Scenario: {base_root} ===")

    # infer paths
    val_path  = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_val_prefix.csv"
    )
    test_path = os.path.join(
        PREFIX_INPUT_FOLDER, f"{base_root}_warm10_test_prefix.csv"
    )

    if not os.path.exists(val_path) or not os.path.exists(test_path):
        print(f"  Skipping: missing val or test for {base_root}")
        return

    # --- load prefix splits ---
    df_train = load_prefix_file(train_path)
    df_val   = load_prefix_file(val_path)
    df_test  = load_prefix_file(test_path)

    print(f"  Train prefixes: {len(df_train)}, cases: {df_train['case_id'].nunique()}")
    print(f"  Val   prefixes: {len(df_val)}, cases: {df_val['case_id'].nunique()}")
    print(f"  Test  prefixes: {len(df_test)}, cases: {df_test['case_id'].nunique()}")

    # --- build case tables (one row per case) ---
    cases_train = build_case_table(df_train)
    cases_val   = build_case_table(df_val)
    cases_test  = build_case_table(df_test)

    print(f"  Train cases: {len(cases_train)}, "
          f"Val cases: {len(cases_val)}, Test cases: {len(cases_test)}")

    if len(cases_train) == 0:
        print("  No train cases -> skip.")
        return

    # --- build training time series for statsforecast ---
    unique_id = base_root  # any string to identify this series
    ts_train  = build_ts_df(cases_train, unique_id=unique_id)

    # --- fit AutoETS on train series ---
    sf = fit_autoets(ts_train)

    # --- forecast cycle times for val + test cases ---
    h_val  = len(cases_val)
    h_test = len(cases_test)
    h_total = h_val + h_test

    if h_total == 0:
        print("  No val/test cases -> nothing to predict.")
        return

    D_hat_all = forecast_cycle_times(sf, h=h_total)
    D_hat_val = D_hat_all[:h_val]
    D_hat_test = D_hat_all[h_val:]

    # case_ids in the same order as cases_val / cases_test
    case_ids_val  = cases_val["case_id"].to_numpy()
    case_ids_test = cases_test["case_id"].to_numpy()

    # --- attach predictions to prefix rows ---
    df_val_ets = add_ets_predictions_to_prefix(
        df_val, cases_val, D_hat_val, case_ids_val
    )
    df_test_ets = add_ets_predictions_to_prefix(
        df_test, cases_test, D_hat_test, case_ids_test
    )

    # --- save outputs ---
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    out_val  = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_val_prefix_ets.csv")
    out_test = os.path.join(OUTPUT_FOLDER, f"{base_root}_warm10_test_prefix_ets.csv")

    df_val_ets.to_csv(out_val, index=False)
    df_test_ets.to_csv(out_test, index=False)

    print(f"  Saved:\n"
          f"    {out_val}  ({len(df_val_ets)} rows)\n"
          f"    {out_test} ({len(df_test_ets)} rows)")


def main():
    if not os.path.isdir(PREFIX_INPUT_FOLDER):
        raise SystemExit(f"Prefix folder not found: {PREFIX_INPUT_FOLDER}")

    # find all *_warm10_train_prefix.csv as "base" scenarios
    train_files = sorted(
        glob.glob(os.path.join(PREFIX_INPUT_FOLDER, "*_warm10_train_prefix.csv"))
    )

    if not train_files:
        raise SystemExit(f"No '*_warm10_train_prefix.csv' files in {PREFIX_INPUT_FOLDER}")

    print(f"Found {len(train_files)} scenario(s) to process.")
    for train_path in train_files:
        process_scenario(train_path)


if __name__ == "__main__":
    main()


c:\Users\990215322\AppData\Local\miniconda3\envs\mupro\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 8 scenario(s) to process.

=== Scenario: log_FIFO_run0_EXP_dedicated_C1 ===
  Train prefixes: 142404, cases: 4491
  Val   prefixes: 47885, cases: 1508
  Test  prefixes: 47885, cases: 1508
  Train cases: 4491, Val cases: 1508, Test cases: 1508
  Saved:
    out/251110/prefix_with_ets\log_FIFO_run0_EXP_dedicated_C1_warm10_val_prefix_ets.csv  (47885 rows)
    out/251110/prefix_with_ets\log_FIFO_run0_EXP_dedicated_C1_warm10_test_prefix_ets.csv (47885 rows)

=== Scenario: log_FIFO_run0_EXP_hybrid30_C1 ===
  Train prefixes: 124515, cases: 4491
  Val   prefixes: 41568, cases: 1500
  Test  prefixes: 41604, cases: 1500
  Train cases: 4491, Val cases: 1500, Test cases: 1500
  Saved:
    out/251110/prefix_with_ets\log_FIFO_run0_EXP_hybrid30_C1_warm10_val_prefix_ets.csv  (41568 rows)
    out/251110/prefix_with_ets\log_FIFO_run0_EXP_hybrid30_C1_warm10_test_prefix_ets.csv (41604 rows)

=== Scenario: log_FIFO_run0_EXP_pooled_C1 ===
  Train prefixes: 119313, cases: 4467
  Val   prefixes: 40040, c

In [1]:
%pip install statsforecast


Note: you may need to restart the kernel to use updated packages.
